# W1D5：代码组织 + 模块化 + 生产化思维**今天要解决的问题**：你写了一周的Python代码，现在一个文件已经几百行了——加新功能要上下翻屏找位置，mentor说要"定时自动跑"但你的代码全在Notebook里。今天你要把代码组织成可维护、可复用、可部署的专业项目。**你要用到的武器**：import机制 + 模块与包 + .py vs .ipynb + 虚拟环境 + PEP 8**需要的前置知识**：D1容器 / D2控制流 / D3函数 / D4文件IO---### 📋 今日知识地图- [ ] import机制：`import module` / `from...import` / `import...as` / 搜索路径 / 文件名陷阱- [ ] 模块与包：一个.py=一个模块 / `__name__=="__main__"` / `__init__.py` / 推荐项目结构- [ ] .py vs .ipynb——生产化思维：各自适用场景 / Notebook探索→.py提取→import- [ ] 虚拟环境：`venv`创建与激活 / `pip install` / `requirements.txt` / 为什么需要隔离- [ ] 代码风格PEP 8：缩进4空格 / 行宽79 / 命名规范 / docstring---

## 🔴 认知冲突：你的代码失控了这一周你写了不少代码——D1的容器操作、D2的订单筛选、D3的函数封装、D4的文件读写清洗。你打开昨天的Notebook，发现它已经300行了。现在mentor小王走过来说："帮我加一个'按品类统计销售额'的功能。"你开始上下翻屏找该把新函数插在哪——翻了10分钟还没找到合适的位置。更头疼的是，小王补了一句：> "对了，下周开始这个分析脚本要每天早上8点自动跑一遍，输出当天的日报。你那个Notebook……能定时跑吗？"你沉默了。Notebook要手动点"Run All"，它没法被系统定时任务调用。而且所有代码挤在一起，改一处怕别处跟着崩。**你刚才遇到的问题，今天要学的"代码组织+模块化+生产化思维"就是专门解决它的。**

## 📦 沙盘衔接**回到速卖通**：昨天你用正则从订单备注里提取了金额、清洗了脏CSV、把结果写回了干净文件。mentor小王看了说"跑得通，不错"。但今天早上他又来了：> "小陈，昨天那个订单清洗的代码，产品组那边也想用。你把你那个Notebook整理一下，拆成几个.py文件，让他们import就能用。另外我刚说了——下周开始每天早上8点自动出日报，你那Notebook没法定时跑。今天把这事搞定啊。"你的任务很明确：把Notebook里的分析代码，变成**可以复用的.py模块** + **可以从命令行执行的脚本**。

---## 概念1：import机制——让代码"找到"别的代码### 🤔 你现有的知识够用吗？你刚写了一个函数 `calc_total(orders)`，它在一个叫 `analyze.py` 的文件里。现在你在另一个文件 `main.py` 里想用它——但Python怎么知道 `calc_total` 在哪儿？如果你在 `main.py` 里直接写 `calc_total(orders)`，Python会报 `NameError: name 'calc_total' is not defined`。**import就是专门解决这个的——它让一个文件"看到"另一个文件里的函数和变量。**### import机制**① 说白了就是**：你电脑里有一份Word文档，想在另一份PPT里引用它的内容——你不需要复制粘贴整篇文档，只需要"引用"一下文件名。import就是Python世界的"引用"——一个文件说"我要用那个文件里的XX"，Python就把XX拿过来。**② 在沙盘里怎么用**：`main.py` 里 `from src.analyze import calc_total`——导入后用法和在自己文件里定义的函数完全一样，但代码只存在一个地方。**③ 跟你D3/D4的关系**：D3你学的是"把重复代码提取成函数"。import是同样的思路升一级："跨文件复用函数"。D4的 `csv.DictReader` 读写逻辑放到 `io.py` 里，然后 `from io import read_orders`。**④ 为什么这样设计**：Python启动时不加载所有代码——import让你**按需加载**。搜索顺序：当前目录 → PYTHONPATH → 标准库 → site-packages。这意味着：**如果你当前目录有 `csv.py`，`import csv` 会导入你的文件而不是标准库！**

In [ ]:
# import的三种姿势import math           # 方式1：导入整个模块，用math.sqrt()调用print(math.sqrt(16))  # 4.0from math import sqrt  # 方式2：只导入sqrt，直接调用print(sqrt(25))        # 5.0import pandas as pd    # 方式3：起别名（行业约定）# pd.read_csv(...)# ⚠️ from math import *  # 不推荐！你不知道导入了什么名字，可能覆盖已有变量

In [ ]:
# import小练：找出bug并修复# 下面这行报错：TypeError: 'float' object is not callablefrom math import sqrtsqrt = sqrt(16)   # 本意：把sqrt(16)的结果存到变量print(sqrt(9))    # 想再算一次——但这行会报错！# 你的诊断：为什么报错？怎么修复？

---## 概念2：模块与包——从单文件到项目结构### 🤔 你现有的知识够用吗？你现在有一个 `orders.py`，里面塞了10个函数：读CSV的、写CSV的、清洗城市名的、转换金额类型的、算总额的……你每次找函数都要从头滚到尾。更麻烦的是：同事只想用你的"读CSV"函数——他却必须import整个300行的文件。**模块化就是专门解决这个的——把10个函数按"做什么"分到3-4个文件里，每个文件只负责一件事。**### 模块与包**① 说白了就是**：你家工具箱——螺丝刀、扳手、锤子各放一个抽屉，而不是全扔在一个大箱子里。每个`.py`文件就是一个"抽屉"（模块），装一类工具。`__init__.py`是抽屉上的标签——告诉Python"这是个工具箱"。**② 在沙盘里怎么用**：拆成 `src/io.py`（文件读写）+ `src/clean.py`（数据清洗）+ `src/analyze.py`（统计分析）。`main.py`做入口，像乐高一样把它们拼起来。**③ 跟你D3学的函数的关系**：D3说"重复代码→提取成函数"，模块化是同样思路上升一级：**相关函数→归到一个文件**。模块 = 函数的容器，就像list = 数据的容器。**④ 为什么这样设计——`__name__ == "__main__"`是关键**：一个.py文件有两种命运：①被直接运行（`python main.py`）→ `__name__` = `"__main__"`；②被import → `__name__` = 模块名。这个判断让你把"测试代码"和"可复用代码"放同一个文件——import时不执行测试，直接运行才执行。没有它，你就要维护两份代码。

In [ ]:
# 模块长这样——一个完整的 analyze.py 示例"""订单分析模块——计算各种统计指标"""def calc_total(orders):    """计算订单总金额"""    return sum(o['金额'] for o in orders)def filter_by_city(orders, city):    """筛选指定城市的订单"""    return [o for o in orders if o['城市'] == city]# 👇 被import时不执行，直接 python analyze.py 时才执行if __name__ == "__main__":    test_orders = [{'金额': 100}, {'金额': 200}]    print(f"测试结果：总金额 = {calc_total(test_orders)}")  # 输出: 300

In [ ]:
# 模块小练：把零散函数按"做什么"分到不同文件# 现有函数：# read_csv(path)        — 读CSV文件# write_csv(data, path) — 写CSV文件# strip_city(name)      — 清洗城市名（去空格）# to_int(value)         — 金额字符串转整数# calc_avg(orders)      — 计算平均订单金额# top_n(orders, n)      — 找金额最高的n笔订单# TODO: 写出你的分组方案# ==== io.py ====# ==== clean.py ====# ==== analyze.py ====

---## 概念3：.py vs .ipynb——生产化思维### 🤔 你现有的知识够用吗？你现在写的所有代码都在Notebook（.ipynb）里。小王让你"每天早上8点自动出日报"——但Notebook要手动点Run All，没法被系统定时任务调用。产品组同事也说："我只想要 `calc_total` 函数，不需要你前面30个探索性cell。"**.py脚本就是专门解决这个的——脱离Jupyter独立运行、可被import、可部署到服务器定时执行。**### .py vs .ipynb——生产化思维**① 说白了就是**：Notebook是你的"实验记录本"——边写边试、画图展示，适合探索和汇报。.py是你的"自动化机器"——干净、可重复执行、可被调度、可团队协作。**② 在沙盘里怎么用**：你在Notebook里探索订单数据→验证分析思路→把核心逻辑提取到 `src/` 的.py文件里。以后每天早上 `python main.py` 一行命令完成昨天手动点20个cell的工作。**③ 跟你D4学的文件IO的关系**：D4的读写逻辑从Notebook的cell"搬"到 `io.py` 后，Notebook和main.py都能 `from io import read_orders`——代码只存一处，多场景复用。**④ 为什么这样设计**：Notebook底层是JSON（为人看优化），.py是纯文本（为机器执行优化）。数据科学标准工作流：Notebook探索（试验田）→ 核心逻辑提取.py（生产车间）→ Notebook import展示结果。**黄金法则：探索用Notebook，生产用.py。**

In [ ]:
# .ipynb → .py 提取演示# 假设你在Notebook的某个cell里探索出了这段清洗逻辑——提取成 clean.py# ==== clean.py 的内容 ===="""数据清洗模块——负责把脏数据洗干净"""def clean_amount(value):    """清洗金额：'299元' → 299"""    if isinstance(value, str):        value = value.replace('元', '').strip()    return int(value)def clean_city(name):    """清洗城市名：去空格"""    return name.strip()# 现在Notebook里只需要一行：from clean import clean_amount, clean_city# crontab可以直接调度：python main.py

In [ ]:
# .py vs .ipynb 小练：判断每种场景该用哪个# 在注释里回答，写 .ipynb 或 .py + 一句话理由# 场景A：第一次拿到新数据，想看分布、画图、试几种分析思路# 场景B：每天早上8点自动拉数据→清洗→生成PDF日报→发邮件# 场景C：给老板做周报演示，需要代码、图表和文字交替展示# 场景D：5个同事各自在自己的项目里调用同一个数据清洗工具

---## 概念4：虚拟环境——给每个项目一个干净的"房间"### 🤔 你现有的知识够用吗？下个月你要同时维护两个项目：项目A用pandas 1.5，项目B用pandas 2.0（API有变化）。如果你只有一个Python环境，装2.0就会覆盖1.5——项目A可能突然跑不通。更糟的是：你把代码发同事，他说"跑不了，报ModuleNotFoundError"——因为你没说明依赖了什么包、什么版本。**虚拟环境就是专门解决这个的——每个项目有自己独立的Python包集合，互不干扰。**### 虚拟环境**① 说白了就是**：你在一个大楼里租了两个房间，每个房间有自己的家具（Python包），互不影响。房间A放pandas 1.5，房间B放pandas 2.0。`requirements.txt`就是"家具清单"——告诉别人怎么复刻你的房间。**② 在沙盘里怎么用**：项目根目录创建venv → 激活 → `pip install pandas` → `pip freeze > requirements.txt`。同事clone代码后 `pip install -r requirements.txt` 一键重建。**③ 跟你本周所学的关系**：D3函数的依赖是代码内部的（函数A调用函数B）；虚拟环境管理的依赖是项目外部的（你的代码依赖第三方库）。内外两层依赖都管好了，项目才能真正"跑在别人机器上"。**④ 为什么这样设计**：Python有40万+第三方包且不断更新。虚拟环境在每个项目里创建独立的 `site-packages` 目录，彻底隔离。代价是多占一点硬盘空间，换来"换一台机器也能跑"的确定性——这是入职第一天就要会的技能。

In [ ]:
# 虚拟环境实战——在终端执行的命令（Jupyter里用 ! 前缀模拟）# 步骤1：创建虚拟环境（项目根目录，只需一次）# !python -m venv venv# 步骤2：激活（Windows PowerShell）# venv\Scripts\activate        → 命令行前面出现 (venv)# 步骤3：安装依赖# !venv\Scripts\python -m pip install pandas# 步骤4：导出依赖清单# !venv\Scripts\python -m pip freeze > requirements.txt# 步骤5：同事重建环境# git clone <项目> → python -m venv venv → 激活 → pip install -r requirements.txtprint("👆 在终端(PowerShell)里逐条执行，不要在Python里执行")

---## 概念5：PEP 8——代码的"着装规范"### 🤔 你现有的知识够用吗？下面两段代码功能完全一样——哪段让你更想读？```python# 版本A: def calcTotalOrders(x):return sum(x);totalRevenue=sum(o['amount']for o in x)# 版本B: def calc_total_orders(orders):#            """计算订单总金额"""#            return sum(o['amount'] for o in orders)```版本A能跑，但一个月后你自己回来看也要盯5秒。**PEP 8不是"教条"——它是Python社区约定俗成的"方言"。写不规范代码，就像说普通话带浓重口音：别人能听懂，但费劲。**### PEP 8——Python代码风格指南**① 说白了就是**：每个圈子有自己的着装规范。Python圈子的规范就是PEP 8——变量怎么命名、缩进用几个空格、一行多长。遵守它，你的代码看起来"像Python"。**② 在沙盘里怎么用**：函数命名用 `snake_case`（`calc_total`✓，`CalcTotal`✗），缩进4空格，每个函数写docstring。**③ 跟你D3学的函数的关系**：D3教你"怎么造工具"，PEP 8教你"造出来的工具该是什么模样"——命名规范、docstring写法、代码排版。**④ 为什么这样设计**：Guido van Rossum说过："代码被阅读的次数远多于被编写的次数。"PEP 8的核心目的是**降低阅读成本**——当所有人的代码遵循同一套视觉规范时，大脑不需要切换模式。4空格缩进不是随便选的：<3空格层级不明显，>4空格行宽浪费。79字符行宽强制你"不要把太多逻辑塞进一行"。

In [ ]:
# PEP 8 规范速查——改前 vs 改后# ❌ 不规范def CalcTotal(Orders):return sum([o['金额']for o in Orders])def getOrdersByCityAndAmount(orders,city,min_amount):    return[o for o in orders if o['城市']==city and o['金额']>min_amount]# ✅ PEP 8 规范def calc_total(orders):    """计算订单总金额"""    return sum(o['金额'] for o in orders)def filter_orders(orders, city, min_amount):    """筛选指定城市且金额大于min_amount的订单"""    return [        o for o in orders        if o['城市'] == city and o['金额'] > min_amount    ]# 差异：snake_case命名 / 逗号后空格 / 操作符两边空格 / docstring / 函数间空两行

---## 🏋️ 递进练习（7题，L2→L4）### 基础练习1：创建你的第一个虚拟环境**场景**：通用 | **覆盖知识点**：虚拟环境-创建/激活/安装/导出创建虚拟环境 → 安装pandas → 验证安装 → 导出依赖清单。用 `!` 前缀在下方code cell执行终端命令（共4步）。**预期输出**：打印出pandas版本号（如 `2.0.3`），生成 `my_requirements.txt` 文件

In [ ]:
# 基础练习1：虚拟环境创建与验证# 步骤1：创建虚拟环境 w1d5_env# 步骤2：用虚拟环境的Python安装pandas# 步骤3：验证pandas安装成功（打印版本号）# 步骤4：导出依赖清单到 my_requirements.txt

### 基础练习2：把D4的订单分析代码拆成模块**场景**：速卖通电商 | **覆盖知识点**：模块与包-项目结构/模块拆分、import机制下面是一段"全塞在一起"的订单分析代码。拆成4个文件：`src/io.py` / `src/clean.py` / `src/analyze.py` / `main.py`。**预期输出**：用 `# ====== 文件名 ======` 标注每个文件的边界，确保import关系正确。

In [ ]:
# 基础练习2：把下面代码拆成模块# 原始代码（全在一个文件里）：# import csv# def read_orders(path):#     with open(path, 'r', encoding='utf-8') as f:#         return list(csv.DictReader(f))# def write_orders(orders, path):#     with open(path, 'w', encoding='utf-8', newline='') as f:#         writer = csv.DictWriter(f, fieldnames=orders[0].keys())#         writer.writeheader()#         writer.writerows(orders)# def clean_amount(value):#     return int(str(value).replace('元', '').strip())# def clean_city(name): return name.strip()# def clean_orders(orders):#     for o in orders:#         o['金额'] = clean_amount(o['金额'])#         o['城市'] = clean_city(o['城市'])#     return orders# def calc_total(orders): return sum(o['金额'] for o in orders)# def calc_avg(orders): return calc_total(orders) / len(orders) if orders else 0# def top_orders(orders, n=3):#     return sorted(orders, key=lambda o: o['金额'], reverse=True)[:n]# orders = read_orders('data/orders.csv')# orders = clean_orders(orders)# print(f"总金额: {calc_total(orders)}, 均价: {calc_avg(orders)}")# TODO: 拆成4个文件，用 # ====== 文件名 ====== 标注边界

### 基础练习3：import跑通全流程**场景**：速卖通电商 | **覆盖知识点**：import-from...import、__name__保护假设 `src/analyze.py` 已有 `calc_total`/`calc_avg`/`top_orders` 三个函数。写 `main.py`：import这些函数 → 用测试数据验证 → 用 `if __name__ == "__main__":` 保护。**预期输出**：`总金额: 1225  |  平均: 245.0  |  TOP3: ['ORD004', 'ORD001', 'ORD002']`

In [ ]:
# 基础练习3：写main.pytest_orders = [    {'订单号': 'ORD001', '商品': '蓝牙耳机', '金额': 299, '城市': '北京'},    {'订单号': 'ORD002', '商品': '连衣裙', '金额': 159, '城市': '上海'},    {'订单号': 'ORD003', '商品': '充电宝', '金额': 89, '城市': '深圳'},    {'订单号': 'ORD004', '商品': '运动鞋', '金额': 599, '城市': '北京'},    {'订单号': 'ORD005', '商品': 'T恤', '金额': 79, '城市': '上海'},]# 模拟 src/analyze.py 中的函数（避免import报错，直接定义）：def calc_total(orders): return sum(o['金额'] for o in orders)def calc_avg(orders): return calc_total(orders) / len(orders) if orders else 0def top_orders(orders, n=3): return sorted(orders, key=lambda o: o['金额'], reverse=True)[:n]# TODO: 写 main.py 的内容# 1. 用 from ... import ... 导入三个函数# 2. 用 if __name__ == "__main__": 保护主逻辑# 3. 输出格式：总金额: 1225  |  平均: 245.0  |  TOP3: ['ORD004', 'ORD001', 'ORD002']

### 进阶练习1：重塑游戏分析Notebook（换行业）**场景**：手游工作室（游戏行业——新场景） | **覆盖知识点**：.py vs .ipynb-最佳实践一个手游Notebook有12个cell：前3个探索+画图，中间6个是装备统计/等级分布/充值分段分析，最后3个重新跑分析导出Excel。产品经理说："以后每周出一份同样报告，能自动化吗？"**预期输出**：写出（1）哪些逻辑→哪些.py文件（2）Notebook保留什么（3）main.py怎么组织

In [ ]:
# 进阶练习1：手游Notebook重构方案# 原始Notebook结构：#   cell 1-3: 读CSV + 画图 + 数据概览（探索部分）#   cell 4-5: 统计玩家装备数量、按等级分组#   cell 6-7: 计算各等级段充值总额、ARPU#   cell 8-9: 筛选高价值玩家（充值>1000）、导出名单#   cell 10-12: 重新画图 + 导出Excel报告# TODO: 写出你的重构方案# - 哪些逻辑→哪个.py文件# - Notebook保留什么# - main.py做什么

### 进阶练习2：让环境可复现（换行业）**场景**：滴滴出行（出行行业——新场景） | **覆盖知识点**：虚拟环境-pip freeze/环境重建行程分析项目发给同事后跑不起来，报 `ModuleNotFoundError`。你需要给他 `requirements.txt`。**预期输出**：依赖清单（至少3行含版本号）+ 同事重建环境的3步操作说明

In [ ]:
# 进阶练习2：环境可复现# 步骤1：导出依赖清单的命令# TODO: 写一行终端命令# 步骤2：同事操作步骤（3步，每步写清命令）# TODO# 步骤3：requirements.txt 内容（至少3行）# TODO

### 挑战练习：重构D4的订单清洗代码**场景**：速卖通电商 | **覆盖知识点**：全部5个概念 + D1-D4把下面的D4订单清洗代码改造成标准项目：至少3个.py模块 + 每个函数docstring + PEP 8规范 + main.py有 `__name__` 保护。**至少试15分钟。卡住了在注释里写下你卡在哪一步、你的思路。**

In [ ]:
# 挑战练习：重构D4的订单清洗代码# 原始代码（来自D4的脏CSV处理）：# import csv, re# def read_csv(path):#     with open(path, 'r', encoding='utf-8') as f:#         return list(csv.DictReader(f))# def clean_row(row):#     row['城市'] = row['城市'].strip()#     amount_str = row['金额']#     match = re.search(r'\d+', str(amount_str))#     row['金额'] = int(match.group()) if match else 0#     return row# def filter_valid(orders):#     return [o for o in orders if o['金额'] > 0 and o['城市']]# def calc_stats(orders):#     total = sum(o['金额'] for o in orders)#     return {'总订单数': len(orders), '总金额': total, '均价': total/len(orders)}# orders = read_csv('orders.csv')# orders = [clean_row(o) for o in orders]# orders = filter_valid(orders)# print(calc_stats(orders))# TODO: 拆成至少3个.py文件 + main.py# 要求：docstring + PEP 8 + __name__保护# 用 # ====== 文件名 ====== 标注边界

### 迁移练习：整理外卖分析"遗产项目"**场景**：美团外卖（全新领域——和沙盘、进阶场景都不同）交接同事发来一个文件夹：`分析v1.ipynb`（80个cell）+ `分析v2-最终.ipynb` + `分析v3-修改.ipynb` + `test.py`（无注释）+ `requirements.txt`（空文件）。**你需要自己判断今天学的哪个武器能解决这个问题。****预期输出**：步骤清单（≥5步），每步标注用到的武器名称

In [ ]:
# 迁移练习：整理外卖分析"遗产项目"# TODO: 写出整理方案（至少5步）# 每步格式：步骤X: [做什么] —— 用到: [武器名称]# 步骤1:# 步骤2:# 步骤3:# 步骤4:# 步骤5:

In [ ]:
# 加分练习A：PEP 8 修复# 下面的代码能跑但风格一团糟——把它改成符合PEP 8的版本# def GetData(path):#     data=[]#     import csv#     with open(path,'r',encoding='utf-8') as f:#         for row in csv.DictReader(f):#             data.append(row)#     return data# def CLEAN_DATA(DATA):#     for D in DATA:#         D['price']=float(D['price'])#         D['city']=D['city'].strip().upper()#     return DATA# TODO: 修复上面的代码——PEP 8命名、空格、docstring、import位置

In [ ]:
# 加分练习B：import故障排查# 小王把项目发给你，你运行 python main.py 报错了。诊断每个场景的问题# 场景1：报错 ModuleNotFoundError: No module named 'analyze'#        main.py里写了 from analyze import calc_total#        但目录结构是：project/src/analyze.py, project/main.py# 你的诊断：# 场景2：报错 AttributeError: module 'json' has no attribute 'load'#        项目里有个文件叫 json.py# 你的诊断：# 场景3：import stats 时报错，说找不到，但 stats.py 明明在 src/ 目录里# 你的诊断：

In [ ]:
# 加分练习C：虚拟环境故障排查# 新同事操作了以下步骤，但最后 import pandas 报 ModuleNotFoundError# 找出他哪一步错了，写出正确的步骤# 同事的操作：# 1. git clone <项目># 2. python -m venv venv# 3. pip install -r requirements.txt# 4. python main.py# 你的诊断：哪一步有问题？正确操作应该是什么？

---## ⚠️ 新手最容易踩的坑### 坑1：文件名和标准库/第三方库同名 → import诡异报错❌ `# 文件名: csv.py  ← 炸弹！``import csv` → 导入的是你自己的csv.py，不是标准库 → `AttributeError`✅ `# 文件名: my_io.py  ← 避开标准库名字``import csv` → 正确导入标准库csv模块**为什么错**：Python先搜当前目录再搜标准库。你的 `csv.py` 挡在前面了。同理避开 `json.py`、`math.py`、`random.py`。### 坑2：循环import → ImportError❌ `analyze.py: from clean import clean_order` ← A依赖B   `clean.py: from analyze import calc_total` ← B又依赖A → 死锁！✅ 让main.py做调度中心——各模块之间不互相import，main.py同时import两者并协调调用顺序。### 坑3：忘了 `if __name__ == "__main__"` → 被import时自动执行❌ 模块底部裸写测试代码 `print(calc_total(test_orders))` → 同事import时这行也执行！✅ 所有"只在直接运行时才做"的代码放 `if __name__ == "__main__":` 块里。### 坑4：把Notebook当生产代码 → 定时任务跑不了 / git merge灾难❌ 把.ipynb发运维说"每天跑这个" / 三个"最终版"Notebook同时存在✅ Notebook探索+展示 | .py生产+定时任务 | 依赖用venv+requirements.txt锁定 | git管理.py文件

---## 📝 参考答案### 概念小练- **import小练**：`sqrt = sqrt(16)` 把函数名覆盖成了数字4.0 → 下一行 `print(sqrt(9))` = `print(4.0(9))` → TypeError。修复：`result = sqrt(16)`- **模块小练**：`io.py`(read_csv, write_csv) + `clean.py`(strip_city, to_int) + `analyze.py`(calc_avg, top_n)- **.py vs .ipynb小练**：A=.ipynb(探索) B=.py(定时) C=.ipynb(展示) D=.py(共享工具)### 基础练习**基础1**：`python -m venv w1d5_env` → `w1d5_env\Scripts\python -m pip install pandas` → 验证版本号 → `pip freeze > my_requirements.txt`**基础2**：`src/io.py`(read_orders, write_orders) + `src/clean.py`(clean_amount, clean_city, clean_orders) + `src/analyze.py`(calc_total, calc_avg, top_orders) + `main.py`(from src.io import... from src.clean import... from src.analyze import... + if __name__保护)**基础3**：```pythonif __name__ == "__main__":    print(f"总金额: {calc_total(test_orders)}  |  "          f"平均: {calc_avg(test_orders)}  |  "          f"TOP3: {[o['订单号'] for o in top_orders(test_orders)]}")```### 进阶练习2`pip freeze > requirements.txt` → 同事：①`git clone` ②`python -m venv venv && 激活` ③`pip install -r requirements.txt && python main.py`requirements.txt: `pandas==2.0.3\nnumpy==1.25.2\nopenpyxl==3.1.2`---## 🔧 偏差诊断| 如果你的输出/报错是... | 你可能的误解是... | 补救 ||----------------------|-----------------|------|| `ModuleNotFoundError: No module named 'src'` | main.py不在项目根目录，或缺`__init__.py` | 确认在根目录运行，检查`src/__init__.py`存在 || `ImportError: cannot import name 'calc_total'` | 函数名写错，或函数定义在`if __name__`里面 | 回到概念2讲解cell——函数定义必须在if判断**外面** || `AttributeError: module 'csv' has no attribute 'DictReader'` | 你的文件名也叫`csv.py`，覆盖了标准库 | 回到坑1——立即重命名文件 || `pip install`报`Access denied` | 没激活venv，试图往系统Python装包 | 回到概念4——先激活venv再pip install || 两个模块互相import报`ImportError` | A import B同时B import A → 循环死锁 | 回到坑2——让main.py做调度中心 || 函数拆对了但import路径不对 | 不理解包=文件夹+`__init__.py` | 回到概念1讲解cell④段"搜索路径"部分 |

---## 🔙 回到冲突：把"面条代码"变成标准项目今天开头你遇到两个问题：300行代码塞在一个文件里 + Notebook没法定时执行。现在用今天学的武器解决它。

In [ ]:
# 回到冲突：改造这段"面条代码"# 小王丢给你一个能跑但乱糟糟的脚本——把它拆成标准项目结构# === 原始代码 ===# import csv, json# with open('orders.csv', 'r', encoding='utf-8') as f:#     reader = csv.DictReader(f)#     orders = list(reader)# for o in orders:#     o['金额'] = int(str(o['金额']).replace('元', '').replace(' ', ''))#     o['城市'] = o['城市'].strip()#     if not o['城市']: o['城市'] = '未知'# total = sum(o['金额'] for o in orders)# beijing_orders = [o for o in orders if o['城市'] == '北京']# beijing_total = sum(o['金额'] for o in beijing_orders)# print(f"总销售额: {total}, 北京: {beijing_total}")# with open('report.json', 'w', encoding='utf-8') as f:#     json.dump({'总销售额': total, '北京销售额': beijing_total}, f, ensure_ascii=False)# TODO: 拆成标准项目结构# 要求：src/io.py + src/clean.py + src/analyze.py + main.py# 每个函数docstring + PEP 8 + main.py用if __name__保护# 用 # ==== 文件名 ==== 标注每个文件的边界

---## ✅ 今天你学会了吗？- [ ] **初级（L2：能用）**：不看笔记，独立完成基础练习1-3  没过 → 回看对应概念的讲解和代码cell，重做基础练习- [ ] **中级（L3：能讲）**：下面写一段200字以内的白话，讲给不懂技术的同事听：        "为什么不能把所有代码写在一个Notebook里？import是干什么的？虚拟环境又是什么？"  *（在此写你的白话解释）*  没过 → 回到概念1-4的讲解cell看第④段- [ ] **高级（L4：能判断）**：迁移练习独立完成了吗？给你一个从没见过的烂摊子项目，         你能自己判断该用今天学的哪些武器来整理它吗？  没过 → 补做迁移练习，不许查笔记---### 📋 概念掌握检查表（对照今日知识地图逐项检验）- [ ] import机制：能写3种import方式、能解释搜索路径、知道为什么文件名不能叫csv.py- [ ] 模块与包：能解释 `__name__ == "__main__"` 作用、知道 `__init__.py` 干什么- [ ] .py vs .ipynb：能说清什么时候用Notebook、什么时候用.py- [ ] 虚拟环境：能创建venv、激活、安装包、导出requirements.txt- [ ] PEP 8：变量名snake_case、缩进4空格、每个函数有docstring- [ ] 项目结构：能画出标准项目目录 + 全部4个陷阱都能识别---**明天**：你将把这周所有武器——容器、控制流、函数、文件IO、模块化——全部用上，独立完成一个完整的电商订单分析项目。mentor小王丢给你三份CSV，从读数据到出报告一条龙搞定，还要用Git管理代码。### 🗺️ 今天的概念在本周知识网络中的位置- **D4 → D5**：D4的文件IO代码封装进 `src/io.py`——从"能读写"升级到"可复用的读写模块"- **D5 → D6**：D6周六项目需要标准项目结构——今天学的模块划分、main.py入口、requirements.txt正是交付物骨架- **D5 → W8**：W8数据工程就是把今天写的 `main.py` 部署到crontab——今天学的"生产化思维"到时候直接复用> 💡 今天是你从"Notebook程序员"到"工程化开发者"的转折点。前四天学"怎么写代码"，今天学"怎么组织代码让别人能用、让机器能跑"——这个思维转变，比任何一个具体语法都重要。